# Set-Up

In [ ]:
# Imports for Generating & Viewing Data #

import h5py
import numpy as np
from scipy.ndimage import label, center_of_mass, gaussian_filter
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

In [ ]:
# Constants #
DATA_DIR = "../Data/"

IMAGE_X = 320
IMAGE_Y = 320

GAUSSIAN_X = 48
GAUSSIAN_Y = 48

SCALE_FACTOR = 8

LABEL_X = IMAGE_X // SCALE_FACTOR
LABEL_Y = IMAGE_Y // SCALE_FACTOR

TRAINING_GROWTHS = 7
VALIDATION_GROWTHS = 2
TEST_GROWTHS = 1
IMAGES_PER_GROWTH = 1000

LEFT_CLIP = 100
RIGHT_CLIP = 200
TOP_CLIP = 30
BOTTOM_CLIP = 270

MIN_X_ROLL = 0
MAX_X_ROLL = 300
MIN_Y_ROLL = 0
MAX_Y_ROLL = 300

THRESHOLD = 0.9

SOFT_LABEL_SIGMA = 0.75

SEED = 0
rng = np.random.default_rng(SEED)

In [ ]:
# Read H5 File #
RHEED_data_file = DATA_DIR + "STO_STO_test6_06292022-standard-compressed.h5"


h5 = h5py.File(RHEED_data_file, "r")
growths = [g for g in h5.keys()]
rng.shuffle(growths)  # Shuffle Growths
training_spots = growths[:TRAINING_GROWTHS]
validation_spots = growths[TRAINING_GROWTHS : TRAINING_GROWTHS + VALIDATION_GROWTHS]
test_spots = growths[
    TRAINING_GROWTHS + VALIDATION_GROWTHS : TRAINING_GROWTHS
    + VALIDATION_GROWTHS
    + TEST_GROWTHS
]

print("Raw Train Data Set:")
raw_train_data_dict = {}
for growth in training_spots:
    indices = rng.choice(h5[growth].shape[0], size=IMAGES_PER_GROWTH, replace=False)
    indices.sort()
    raw_train_data_dict[growth] = np.expand_dims(h5[growth][indices], -1).astype(
        np.float32
    )
    print(f"[Growth]: {growth:<25}, [Shape]: {raw_train_data_dict[growth].shape}")

print("Raw Validation Data Set:")
raw_validation_data_dict = {}
for growth in validation_spots:
    indices = rng.choice(h5[growth].shape[0], size=IMAGES_PER_GROWTH, replace=False)
    indices.sort()
    raw_validation_data_dict[growth] = np.expand_dims(h5[growth][indices], -1).astype(
        np.float32
    )
    print(f"[Growth]: {growth:<25}, [Shape]: {raw_validation_data_dict[growth].shape}")

print("Raw Test Data Set:")
raw_test_data_dict = {}
for growth in test_spots:
    indices = rng.choice(h5[growth].shape[0], size=IMAGES_PER_GROWTH, replace=False)
    indices.sort()
    raw_test_data_dict[growth] = np.expand_dims(h5[growth][indices], -1).astype(
        np.float32
    )
    print(f"[Growth]: {growth:<25}, [Shape]: {raw_test_data_dict[growth].shape}")

In [ ]:
# Aggregate Data #

raw_train_data = np.concatenate(list(raw_train_data_dict.values()))
raw_validation_data = np.concatenate(list(raw_validation_data_dict.values()))
raw_test_data = np.concatenate(list(raw_test_data_dict.values()))

print(f"[Train Data Set Shape]: {raw_train_data.shape}")
print(f"[Validation Data Set Shape]: {raw_validation_data.shape}")
print(f"[Test Data Set Shape]: {raw_test_data.shape}")

In [ ]:
# Pad images 300 x 300 -> 320 x 320 #

pad_h = (320 - 300) // 2  # 10 pixels each side
pad_w = (320 - 300) // 2  # 10 pixels each side

pad_config = ((0, 0), (pad_h, pad_h), (pad_w, pad_w), (0, 0))

raw_train_data = np.pad(raw_train_data, pad_config, mode="edge")
raw_validation_data = np.pad(raw_validation_data, pad_config, mode="edge")
raw_test_data = np.pad(raw_test_data, pad_config, mode="edge")

print(f"[Train Data Set Shape]:      {raw_train_data.shape}")
print(f"[Validation Data Set Shape]: {raw_validation_data.shape}")
print(f"[Test Data Set Shape]:       {raw_test_data.shape}")

In [ ]:
# Utility Functions for Processing Data #
def gen_data(num_images: int, dataset, transform=True) -> tuple:
    img_arr = np.empty((num_images, IMAGE_Y, IMAGE_X, 1), dtype=np.float32)
    label_arr = np.empty((num_images, LABEL_Y, LABEL_X, 1), dtype=np.float32)

    for i in tqdm(range(num_images)):
        img, label = img_gen(dataset, transform)
        img_arr[i] = img.astype(np.float32) / 256.0
        label_arr[i] = label.astype(np.float32) / 256.0

    return (img_arr, label_arr)


def img_gen(dataset, transform=True) -> tuple:
    index = np.random.randint(low=0, high=dataset.shape[0])
    img = dataset[index]
    img_label = orig_label(img)

    if transform:
        img, img_label = img_shift(img, img_label)

    img_label = (img_label * 255).astype(np.uint8)
    img = img.astype(np.uint8)

    return (img, img_label)


def orig_label(img) -> np.ndarray:
    img_2d = img.squeeze(-1)  # (320, 320, 1) -> (320, 320)
    cropped = img_2d[TOP_CLIP:BOTTOM_CLIP, LEFT_CLIP:RIGHT_CLIP]
    smoothed = gaussian_filter(np.log1p(cropped), sigma=2)
    threshold = np.percentile(smoothed, 85)
    binary = smoothed > threshold
    labeled, num_features = label(binary)
    centers = center_of_mass(smoothed, labeled, range(1, num_features + 1))

    params = []
    for y, x in centers:  # correctly unpacking 2D coordinates
        gy = (y + TOP_CLIP) * (1.0 / SCALE_FACTOR)
        gx = (x + LEFT_CLIP) * (1.0 / SCALE_FACTOR)
        params.append((gx, gy, 1.0, 1.0, 0.0, 1.0))

    img_label = make_soft_label(
        params=params,
        label_h=LABEL_Y,
        label_w=LABEL_X,
        scale_factor=1.0,
        sigma=SOFT_LABEL_SIGMA,
    )

    return img_label


def img_shift(img, img_label) -> tuple:
    x_roll = np.random.randint(low=MIN_X_ROLL, high=MAX_X_ROLL)
    y_roll = np.random.randint(low=MIN_Y_ROLL, high=MAX_Y_ROLL)

    img = np.roll(img, y_roll, axis=0)
    img = np.roll(img, x_roll, axis=1)

    dy = int(y_roll // SCALE_FACTOR)
    dx = int(x_roll // SCALE_FACTOR)
    label_shifted = np.roll(img_label, shift=(dy, dx), axis=(0, 1))

    return (img, label_shifted)


def make_soft_label(
    params, label_h, label_w, scale_factor, sigma=SOFT_LABEL_SIGMA
) -> np.ndarray:
    yy, xx = np.meshgrid(np.arange(label_h), np.arange(label_w), indexing="ij")
    label = np.zeros((label_h, label_w), dtype=np.float32)

    for center_x, center_y, std_x, std_y, theta, intensity in params:
        gx = center_x / scale_factor
        gy = center_y / scale_factor
        bump = np.exp(-((xx - gx) ** 2 + (yy - gy) ** 2) / (2 * sigma**2))
        label = np.maximum(label, bump)

    return label[..., None]

# Train / Load Qkeras Model

In [ ]:
# Imports for Training #

import tensorflow as tf
import keras.backend as K
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Add, Activation
from tensorflow.keras.activations import relu, sigmoid
from tensorflow.keras.models import Model
from qkeras import QConv2D, QActivation
from qkeras.quantizers import quantized_bits, quantized_relu

In [ ]:
# Create TF Datasets #
# TODO: look into using tf.data.Dataset.from_generator
TRAINING_DATASET_SIZE = 10000
VALIDATION_DATASET_SIZE = 1000
TEST_DATASET_SIZE = 1000
BATCH_SIZE = 50


train_img_arr, train_label_arr = gen_data(TRAINING_DATASET_SIZE, raw_train_data)
val_img_arr, val_label_arr = gen_data(VALIDATION_DATASET_SIZE, raw_validation_data)
test_img_arr, test_label_arr = gen_data(
    TEST_DATASET_SIZE, raw_test_data, transform=False
)

train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_img_arr, train_label_arr))
    .shuffle(TRAINING_DATASET_SIZE, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    tf.data.Dataset.from_tensor_slices((val_img_arr, val_label_arr))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# Qkeras Model Architecture #
TOTAL_BITS = 8
INTEGER_BITS = 0


w_quant = quantized_bits(
    TOTAL_BITS, INTEGER_BITS, symmetric=False, keep_negative=True, alpha=1
)
relu_quant = quantized_relu(TOTAL_BITS, INTEGER_BITS)

input_layer = Input(shape=(IMAGE_Y, IMAGE_X, 1))

x = QConv2D(
    filters=4,
    kernel_size=3,
    strides=2,
    padding="same",
    use_bias=False,
    name="qconv2d_1_1",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(input_layer)
# x = BatchNormalization(name="b_1_1")(x)
x = QActivation(relu_quant, name="qact_1_1")(x)
y = QConv2D(
    filters=4,
    kernel_size=3,
    padding="same",
    use_bias=False,
    name="qconv2d_1_2",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
y = BatchNormalization(name="b_1_2")(y)
x = Add(name="add_1")([x, y])
# x = QActivation(relu_quant, name="qact_1_2")(x)

x = QConv2D(
    filters=6,
    kernel_size=3,
    strides=2,
    padding="same",
    use_bias=False,
    name="qconv2d_2_1",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
# x = BatchNormalization(name="b_2_1")(x)
x = QActivation(relu_quant, name="qact_2_1")(x)
y = QConv2D(
    filters=6,
    kernel_size=3,
    padding="same",
    use_bias=False,
    name="qconv2d_2_2",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
y = BatchNormalization(name="b_2_2")(y)
x = Add(name="add_2")([x, y])
# x = QActivation(relu_quant, name="qact_2_2")(x)

x = QConv2D(
    filters=8,
    kernel_size=3,
    strides=2,
    padding="same",
    use_bias=False,
    name="qconv2d_3_1",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
# x = BatchNormalization(name="b_3_1")(x)
x = QActivation(relu_quant, name="qact_3_1")(x)
y = QConv2D(
    filters=8,
    kernel_size=3,
    padding="same",
    use_bias=False,
    name="qconv2d_3_2",
    kernel_quantizer=w_quant,
    kernel_initializer="lecun_uniform",
)(x)
y = BatchNormalization(name="b_3_2")(y)
x = Add(name="add_3")([x, y])
# x = QActivation(relu_quant, name="qact_3_2")(x)

x = Conv2D(8, 1, padding="same", use_bias=True, name="qconv2d_4")(x)
x = Activation(relu, name="qact_4")(x)

x = Conv2D(1, 1, padding="same", use_bias=True, name="qconv2d_5")(x)
x_prob = Activation(sigmoid, name="sigmoid_out")(x)

baby_yolo_qk = Model(inputs=input_layer, outputs=x_prob, name="baby_yolo")

In [ ]:
# Tensorflow Functions #
POS_WEIGHT = 5.0
NEG_WEIGHT = 1.0


def loss_wbce(y_true, y_pred):

    y_true = tf.cast(y_true, tf.float32)

    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

    bce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    weights = y_true * POS_WEIGHT + (1.0 - y_true) * NEG_WEIGHT

    return tf.reduce_mean(weights * bce)

In [ ]:
# Train or Load #
# TODO: Fix Load
TRAIN_MODEL = True
LOAD_MODEL = False

MODEL_NAME = ""

NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 1e-4


if TRAIN_MODEL and LOAD_MODEL:
    print("Are you sure you want to Train & Load ?")

elif TRAIN_MODEL:
    adam_optimizer = tf.keras.optimizers.Adam(
        learning_rate=1e-3,
        global_clipnorm=1.0,
    )

    baby_yolo_qk.compile(
        optimizer=adam_optimizer,
        loss=loss_wbce,
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=MIN_DELTA,
        restore_best_weights=True,
        verbose=1,
    )

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1,
    )

    history = baby_yolo_qk.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=NUM_EPOCHS,
        callbacks=[early_stop, reduce_lr],
        verbose=1,
    )

elif LOAD_MODEL:
    pass

baby_yolo_qk.summary()

In [ ]:
# Save Model #
SAVE_MODEL = False

if SAVE_MODEL:
    baby_yolo_qk.save(f"Models/{MODEL_NAME}")

# Test QKeras Model

In [ ]:
# Utility Functions for Viewing Data #
# TODO: add comparison functions

In [ ]:
# Predict on Test Set #

predictions_qk = baby_yolo_qk.predict(test_img_arr)

print(f"[Test Shape]: {test_label_arr.shape}")
print(f"[Prediction Shape]: {predictions_qk.shape}")

# Convert to HLS

In [ ]:
# Imports for HLS #

import hls4ml
from hls4ml.utils import config_from_keras_model
from pprint import pprint

import os

xilinx_vitis = "/home/tools/Xilinx/2025/2025.2/Vitis"
xilinx_viv = "/home/tools/Xilinx/2025/2025.2/Vivado"
xilinx_hls = "/home/tools/Xilinx/2025/2025.2/Vitis"

# prepend Vitis bin directory to PATH
# os.environ["PATH"] = f"{xilinx_vitis}/bin:" + os.environ["PATH"]

os.environ["XILINX_HLS"] = xilinx_hls
os.environ["XILINX_VITIS"] = xilinx_vitis
os.environ["XILINX_VIVADO"] = xilinx_viv

In [ ]:
# Utils for HLS #

data = test_img_arr[:3].reshape(3, -1)
np.savetxt("Utils/Real/tb_input_features.dat", data, fmt="%.6f")

data = test_label_arr[:3].reshape(3, -1)
np.savetxt("Utils/Real/tb_output_predictions.dat", data, fmt="%.6f")

In [ ]:
# HLS4ML Config #
# TODO: look into pipeline
# TODO: verify config quantization

config = config_from_keras_model(baby_yolo_qk, granularity="name", backend="Vitis")

# Fifo Depth Optimization (greatly reduces BRAM)
# config["Flows"] = ["vitis:fifo_depth_optimization"]
# hls4ml.model.optimizer.get_optimizer("vitis:fifo_depth_optimization").configure(
#     profiling_fifo_depth=1000
# )

# Strategy
config["Model"]["Strategy"] = "Latency"
# config['Model']['PipelineStyle'] = 'pipeline'

# Reuse Factor
# config["Model"]["ReuseFactor"] = 4
# for layer_cfg in config["LayerName"].values():
#     if "ReuseFactor" in layer_cfg:
#         layer_cfg["ReuseFactor"] = 4

pprint(config)

In [ ]:
# Compile #
# Make sure you copy over the TB Data found in Utils for FOLO Depth Opt.

hls_model = hls4ml.converters.convert_from_keras_model(
    baby_yolo_qk,
    hls_config=config,
    output_dir="folo_hls4ml_Q_Real",
    io_type="io_stream",  # io_parallel
    clock_period=2.0,
    clock_uncertainty="12.5%",
    backend="Vitis",
    part="xcku035-fbva676-2-e",
    project_name="folo",
)

hls_model.compile()

In [ ]:
# Predict #

predictions_hls4ml = hls_model.predict(test_img_arr).reshape(
    TEST_DATASET_SIZE, LABEL_Y, LABEL_X
)

print(f"[Test Shape]: {test_label_arr.shape}")
print(f"[Prediction Shape]: {predictions_hls4ml.shape}")

In [ ]:
# Visually Compare Predictions #
index = np.random.randint(low=0, high=TEST_DATASET_SIZE)

fig, axes = plt.subplots(1, 3)
im0 = axes[0].imshow(test_label_arr[index], cmap="viridis", interpolation="none")
axes[0].set_title("Test Label")
axes[0].axis("off")

im1 = axes[1].imshow(predictions_qk[index], cmap="viridis", interpolation="none")
axes[1].set_title("QK Prediction")
axes[1].axis("off")

im2 = axes[2].imshow(predictions_hls4ml[index], cmap="viridis", interpolation="none")
axes[2].set_title("HLS Prediction")
axes[2].axis("off")

plt.show()

In [ ]:
# Build Model #
# vitis_hls build_prj.tcl
# vitis-run --mode hls --tcl build_prj.tcl

# hls_model.build()